# 2. Preprocessing — NSL-KDD Intrusion Detection

**Goal:** group the 39 specific attack labels into 4 standard attack categories, and turn text columns into numbers the model can use.

**Input:** `nsl_kdd_train_raw.csv`, `nsl_kdd_test_raw.csv` (from notebook 1)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv("data/nsl_kdd_train_raw.csv")
test = pd.read_csv("data/nsl_kdd_test_raw.csv")

print(train.shape, test.shape)

(125973, 43) (22544, 43)


## Group attack labels into 4 categories

39 specific attack names is too granular — some have only 2-3 training examples. Security research groups them into 4 standard families. These groupings come from the original NSL-KDD documentation and follow-up literature.

In [2]:
DOS = {"neptune","back","land","pod","smurf","teardrop","mailbomb","processtable","udpstorm","apache2","worm"}
PROBE = {"satan","ipsweep","nmap","portsweep","mscan","saint"}
R2L = {"guess_passwd","ftp_write","imap","phf","multihop","warezmaster","warezclient",
       "spy","xlock","xsnoop","snmpguess","snmpgetattack","httptunnel","sendmail","named"}
U2R = {"buffer_overflow","loadmodule","rootkit","perl","sqlattack","xterm","ps"}

def categorize(label):
    if label == "normal": return "normal"
    if label in DOS: return "DoS"
    if label in PROBE: return "Probe"
    if label in R2L: return "R2L"
    if label in U2R: return "U2R"
    return "unknown"

train["attack_category"] = train["label"].apply(categorize)
test["attack_category"] = test["label"].apply(categorize)

print("Any 'unknown' categories? (should be 0):")
print("train:", (train["attack_category"] == "unknown").sum())
print("test:", (test["attack_category"] == "unknown").sum())

Any 'unknown' categories? (should be 0):
train: 0
test: 0


In [3]:
train["attack_category"].value_counts()

attack_category
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

**What to notice:** `U2R` has only 52 examples in the full training set — versus 67,343 for `normal`. This severe imbalance is why we'll need `class_weight="balanced"` when training models later.

## Track which attack types are "novel" (unseen in training)

We identified these in notebook 1 — mark them on the test set so we can evaluate performance on them separately later.

In [4]:
train_labels = set(train["label"].unique())
test_labels = set(test["label"].unique())
novel_attacks = test_labels - train_labels

test["is_novel_attack_type"] = test["label"].isin(novel_attacks).astype(int)
print(f"Novel attack rows in test set: {test['is_novel_attack_type'].sum()} out of {len(test)}")

Novel attack rows in test set: 3750 out of 22544


## Encode categorical (text) columns

Three columns — `protocol_type`, `service`, `flag` — are text. Models need numbers.

We fit the encoder on the **training set only** (to avoid leaking test information), then apply it to both. Any `service`/`flag` value in the test set that never appeared in training gets mapped to a safe `"unseen"` category instead of crashing.

In [5]:
categorical_cols = ["protocol_type", "service", "flag"]

for col in categorical_cols:
    le = LabelEncoder()
    le.fit(train[col])
    known = set(le.classes_)

    # map any test-only categories to "unseen"
    test[col + "_safe"] = test[col].apply(lambda x: x if x in known else "unseen")

    # build a combined encoder that includes "unseen" as a valid class
    all_classes = list(le.classes_) + (["unseen"] if "unseen" not in le.classes_ else [])
    le2 = LabelEncoder()
    le2.fit(all_classes)

    train[col + "_enc"] = le2.transform(train[col])
    test[col + "_enc"] = le2.transform(test[col + "_safe"])

print("Encoding done. Example:")
train[["protocol_type", "protocol_type_enc", "service", "service_enc"]].head()

Encoding done. Example:


,protocol_type,protocol_type_enc,service,service_enc
0,tcp,1,ftp_data,20
1,udp,2,other,44
2,tcp,1,private,49
3,tcp,1,http,24
4,tcp,1,http,24


## Build the final feature list

We drop the original text columns (now redundant with their `_enc` versions) and non-feature columns like `label` and `difficulty`.

In [6]:
feature_cols = [c for c in train.columns if c not in
                ["label", "attack_category", "difficulty",
                 "protocol_type", "service", "flag"] and not c.endswith("_safe")]

print(f"Final feature count: {len(feature_cols)}")
print(feature_cols)

Final feature count: 41
['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'protocol_type_enc', 'service_enc', 'flag_enc']


## Save processed data for the modeling notebook

In [7]:
train_out = train[feature_cols + ["label", "attack_category"]]
test_out = test[feature_cols + ["label", "attack_category", "is_novel_attack_type"]]

train_out.to_csv("data/train_processed.csv", index=False)
test_out.to_csv("data/test_processed.csv", index=False)

print("Saved train_processed.csv:", train_out.shape)
print("Saved test_processed.csv:", test_out.shape)

Saved train_processed.csv: (125973, 43)
Saved test_processed.csv: (22544, 44)


## What's next

Open **`03_model_training.ipynb`** to train and compare Logistic Regression, Decision Tree, and Random Forest models.